[finetune](https://github.com/OpenGVLab/VideoMAEv2/blob/master/models/modeling_finetune.py#L198)

In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../../')

In [2]:
from pathlib import Path

import torch
import numpy as np
from PIL import Image

%load_ext autoreload
%autoreload 2
    
from computer_vision.video_mae.parameter_parser import parser
from computer_vision.video_mae.dataset.pretrained_datasets import HybridVideoMAE, DataAugmentationForVideoMAEv2
# from computer_vision.video_mae.models.modeling_pretrain import pretrain_videomae_large_patch16_224
from computer_vision.video_mae.models.modeling_finetune import MLP, Attention, CosAttention, Block, PatchEmbed, trunc_normal_, \
get_sinusoid_encoding_table, VisionTransformer, _cfg

In [3]:
from functools import partial

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp



In [4]:
n_pos=(16//2)*(224//16)*(224//16)
d_model=768
pos_embed=get_sinusoid_encoding_table(n_pos, d_model)
print(f"{pos_embed.shape}, ({pos_embed.min().item():.3f},{pos_embed.max().item():.3f})")
trunc_normal_(pos_embed, 0., 0.0001)
print(f"{pos_embed.shape}, ({pos_embed.min().item():.3f},{pos_embed.max().item():.3f})")

torch.Size([1, 1568, 768]), (-1.000,1.000)
torch.Size([1, 1568, 768]), (-0.000,0.000)


In [5]:
x=torch.rand(10, 30, 80)
module=MLP(in_features=80, hidden_features=40, out_features=10, act_layer=nn.GELU, drop=0.5)
out=module(x)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

out.shape=torch.Size([10, 30, 10])


In [6]:
attn_module=Attention(dim=768, num_heads=8, qkv_bias=True, qk_scale=None, attn_drop=0.1, proj_drop=0.1, attn_head_dim=None)
x=torch.rand(32,196,768)
out=attn_module(x)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

out.shape=torch.Size([32, 196, 768])


In [7]:
attn_module=CosAttention(dim=768, num_heads=8, qkv_bias=True, qk_scale=None, attn_drop=0.1, proj_drop=0.1, attn_head_dim=None)
x=torch.rand(32,196,768)
out=attn_module(x)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

out.shape=torch.Size([32, 196, 768])


In [8]:
# assuming mid-to-late block
block=Block(dim=768, num_heads=12, mlp_ratio=4., qkv_bias=True, drop_path=0.1, init_values=1e-6, act_layer=nn.GELU,
                 norm_layer=nn.LayerNorm, attn_head_dim=None, cos_attn=False)
x=torch.rand(32,196,768)
out=block(x)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

out.shape=torch.Size([32, 196, 768])


In [9]:
patch_embedding=PatchEmbed(img_size=224, patch_size=16, in_chans=3, embed_dim=768, num_frames=16, tubelet_size=2)
x=torch.rand(4,3,16,224,224)
out=patch_embedding(x)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

out.shape=torch.Size([4, 1568, 768])


In [10]:
# vit_small_patch16_224
model=VisionTransformer(img_size=224, patch_size=16, num_classes=1000, embed_dim=384, depth=12, num_heads=6, mlp_ratio=4., qkv_bias=True,
                        drop_path_rate=0., norm_layer=partial(nn.LayerNorm,eps=1e-6), init_values=0., use_learnable_pos_emb=False, init_scale=0.,  
                        all_frames=16, tubelet_size=2, use_mean_pooling=True, with_cp=False, cos_attn=False)
model.default_cfg = _cfg()
x=torch.rand(4,3,16,224,224)
out=model(x)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

out.shape=torch.Size([4, 1000])
